# Interactive Dash: Understanding Callbacks

## Overview 

A large part of Dash is the reactive nature of the application - you can set it up so that when users change values in your application, the results change as well. In this section, we will go over how to use Dash callbacks in some simple settings, then increasingly in more complex settings to allow the user to control plotting results. 

## Understanding Callbacks

We will start with a simple example (though it looks complex!). The application we will run is: `Apps\03_BasicCallbacks`

This is an application that will take user input (a number), then square the number, and return the result. This is a simple app! But there is a fair bit of standard code you will recognize.

Here is the application in full: 

```
# -*- coding: utf-8 -*-
# This is the template for running an app. 

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

# identify external stylesheet if needed (copied straight from Dash docs)
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']


# initialize application
app = dash.Dash(__name__,
                external_stylesheets=external_stylesheets)

# declare app layout
app.layout = html.Div([
    html.H1("Simple Callback example script."),
    html.Br(),
    html.Div(["Input a number: ",
              dcc.Input(id='my-input', 
                        value=1, 
                        type='number')]),
    html.Br(),
    html.Div(['The square of the number is: ',
              html.Div(id='my-output')
              ]
             )
    ]
    )

# add callbacks
@app.callback(
    Output(component_id='my-output', component_property='children'),
    Input(component_id='my-input', component_property='value')
)
def calculate_square(input_value):
    result1 = input_value*input_value
    return result1

# run application
if __name__ == '__main__':
    app.run_server(debug=True)
```


**END OF APPLICATION**


Notice a few changes from our prior apps:

When we declare the final html.Div here:

```
html.Div(['The square of the number is: ',
              html.Div(id='my-output')
              ]
```

There are no children - there is only an `id='my-output'`. This is because we will be passing in the children value through the callback down here:

```
@app.callback(
    Output(component_id='my-output', component_property='children'),
    Input(component_id='my-input', component_property='value')
)
```

Notice that the output identifies the component property of children. 


## About that callback:

let's explore the callback some more. 

```
# add callbacks
@app.callback(
    Output(component_id='my-output', component_property='children'),
    Input(component_id='my-input', component_property='value')
)
def calculate_square(input_value):
    result1 = input_value*input_value
    return result1
```


The callback decorator (`@app.callback`) includes the Output and the Input (in that order) as arguments. Then, immediately after these declarations (no empty line allowed), the code specifies the function that is to be run every time the callback is triggered is defined. 


You can name this function whatever you want. Notice the usual rules of functions apply here – but you do not have to match the named argument of the function to the actual name of the input value. That value will be passed in from the input as defined in the callback decorator.  

Anytime the input changes, the callback function will be called. 


## Callback with plot example, multiple inputs - generate a normal distribution.

I will now walk you through an app with multiple inputs and a single callback, with a graph dependent on the callback behavior. 

Please open this app: `Apps/04_BasicCallbacks_figure_multiple`

Here is the app in full:

```
# -*- coding: utf-8 -*-

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px
import numpy as np


# identify external stylesheet if needed (copied straight from Dash docs)
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']

# initialize application
app = dash.Dash(__name__,
                external_stylesheets=external_stylesheets)

# declare app layout
app.layout = html.Div([
    html.H1("Normal Distribution Explorer"),
    html.Br(),
    dcc.Graph(id='normal-plot'),
    html.P("Select a Mean:"),
    dcc.Slider(id="mean", 
               min=-4,
               max=4,
               value=0, 
               marks={-4: '-4', 0: '0' ,4: '4'}),
    html.P("Select Standard Deviation:"),
    dcc.Slider(id="std", min=1, max=3, value=1, 
               marks={1: '1', 2:'2', 3: '3'})
    ]
    )

# add callbacks
@app.callback(
    Output("normal-plot", "figure"), 
    [Input("mean", "value"), 
     Input("std", "value")])
def create_graph(mean_1, std_1):
    data = np.random.normal(mean_1, std_1, size=500)
    title = "Normal distribution with mean " + str(mean_1) + " and stdev " + str(std_1)
    fig = px.histogram(data, nbins=30, range_x=[-12, 12], 
                       title=title)
    return fig

# run application
if __name__ == '__main__':
    app.run_server(debug=True)
```



## Stateless Server Concepts

We should talk for a moment about the concept of stateless server. In both of our examples above, we never modified a global variable using the user input. The server should never have to keep track of user variables in general. In the figure example, we generated the figure based on user input - we never needed to modify a global variable based on user input. 

If there are multiple users, this is especially important - we don't want them to interfere with each others' sessions.

##  Linking Callbacks to Data Elements

A common task would be to link data set filters to the callbacks. However, if we are going to do this, we need to pre-calculate whatever it is we want to filter within the data set on so that we can then manipulate those unique values in the layout. In the next application, I will walk you through how to do that. 

We will replicate one of our nice plotly interactive graphics from the last session using the scatter plot that was animated. Our goal is to allow the user to select what continent they wish to view, then generate the appropriate animated graph. 

Please open the following application and we will walk though it: `Apps\05_Gapminder_Callbacks`


Here is the full application file:

```
# -*- coding: utf-8 -*-
# This is the template for running an app. 

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px # plotly high level API module
import pandas as pd


# identify external stylesheet if needed (copied straight from Dash docs)
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']


# import data
df_gapminder = \
    pd.read_csv("data/gapminder/gapminder.csv")
continents = df_gapminder['continent'].unique()

# initialize application
app = dash.Dash(__name__,
                external_stylesheets=external_stylesheets)

# declare app layout
app.layout = html.Div(
    id="app-container",
    children=[
        html.H1('Gapminder Data on Life expectancy and gdp'),
        dcc.Dropdown(id='continent-in',
                     options=[{'label': i, 'value': i} for i in continents],
                     value=continents[0]),
        dcc.Graph(
            id='fig-life-exp')
        ]
   )

# add callbacks 
@app.callback(
    Output("fig-life-exp", "figure"), 
    Input("continent-in", "value"))
def create_graph(continent_in):
    # filter data
    filtered_df = df_gapminder[df_gapminder['continent']==continent_in]
    # add animation 
    fig = px.scatter(filtered_df,
           x="gdpPercap", 
           width=800,
           hover_name="country",
           color='country',
           y="lifeExp",
           animation_frame="year", 
           animation_group="country",
           size='pop',
           log_x=True,
           range_x=[filtered_df['gdpPercap'].min()-300,
                    filtered_df['gdpPercap'].max()+1000],
           range_y=[filtered_df['lifeExp'].min()-5,
                    filtered_df['lifeExp'].max()+5])
    return fig


# run application
if __name__ == '__main__':
    app.run_server(debug=True)
```

We demonstrated some nice code patterns in this application. We used pandas for data input and then generated a collection of unique options that we later used in the layout. Inside the layout we used the dropdown with a list comprehension to generate a list of dictionaries for the options based on that unique values list of continents we generated earlier. Within the callbacks, we first changed the data frame in the scope of the function period since we reference this data frame a couple of times. We overtly first filtered the data frame. We want to avoid modifying variables outside the scope of the callback function. 

## More than one Output

The application call back can have multiple outputs. If so, then we return a tuple of items from the function and they are unpacked automatically to the outputs. Here is an example extending the application we just wrote to generate multiple plots, all affected by the user input. 

Please open: `Apps\06_Gapminder_Callbacks_multiple`

here is the entire application:

```
# -*- coding: utf-8 -*-
# This is the template for running an app. 

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px # plotly high level API module
import pandas as pd


# identify external stylesheet if needed (copied straight from Dash docs)
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']


# import data
df_gapminder = \
    pd.read_csv("data/gapminder/gapminder.csv")
continents = df_gapminder['continent'].unique()

# initialize application
app = dash.Dash(__name__,
                external_stylesheets=external_stylesheets)

# declare app layout
app.layout = html.Div(
    id="app-container",
    children=[
        html.H1('Gapminder Data on Life expectancy and gdp'),
        html.P('Please select a contininent:'),
        dcc.Dropdown(id='continent-in',
                     options=[{'label': i, 'value': i} for i in continents],
                     value=continents[0]),
        dcc.Graph(
            id='fig-life-exp'),
        dcc.Graph(
            id='fig-boxplot')
        ]
   )

# add callbacks 
@app.callback(
    Output("fig-life-exp", "figure"), 
    Output("fig-boxplot", "figure"),
    Input("continent-in", "value"))
def create_graph(continent_in):
    # filter data
    filtered_df = df_gapminder[df_gapminder['continent']==continent_in]
    # add animation 
    fig1 = px.scatter(filtered_df,
           x="gdpPercap", 
           width=1000,
           hover_name="country",
           color='country',
           y="lifeExp",
           animation_frame="year", 
           animation_group="country",
           size='pop',
           log_x=True,
           range_x=[filtered_df['gdpPercap'].min(),
                    filtered_df['gdpPercap'].max()],
           range_y=[filtered_df['lifeExp'].min(),
                    filtered_df['lifeExp'].max()])
    fig2 = px.box(filtered_df,
                  width=1000,
                  x='country',
                  y='lifeExp')
    return fig2, fig1


# run application
if __name__ == '__main__':
    app.run_server(debug=True)
```

## Advanced Layout Options

We have not yet discussed advanced layout options beyond spending lots of time understanding and customizing your HTML and CSS code. 

You can use the Bootstrap components library to perform some more advanced layout options with good defaults to choose from: https://dash-bootstrap-components.opensource.faculty.ai/ 

## Deploying Dash

Deploying open source Dash is not trivial. The recommended service in the docs is Heroku, which is a cloud-based app deployment platform. Of course, one of the challenges in deploying data-driven apps is also distributing the  data-- but these questions are beyond the scope of this lesson. We will not be covering how to deploy Dash in this module, but you can check these links out for more information. You must sign up for a Heroku account and have Git up and running as well. 

Of course, if you have paid for Dash Enterprise, your deployment options are much easier (that is a large part of what you are paying for!)

However, as you have seen in these notes you can create a rich Dash app that you can deploy and demonstrate from your laptop for local presentations. You could also bundle the folder containing your app and data for sharing within in an organization-- but this assumes that the recipient will has the necessary Python packages installed. 

https://www.heroku.com/

https://dash.plotly.com/deployment


## Summary:

In this module, you have learned the basics of how Dash callbacks work. You implemented simple and not so simple callbacks, including multiple inputs and multiple outputs. 

We closed the module with a brief discussion of advanced layout options, as well as how to learn to deploy Dash. 